# 09 business simulation

All success, costs and margins are assumptions; a ranking win is not proof of saved revenue.

Reusable calculations live in src/ and run_pipeline.py. This notebook reads executed artifacts so analysis cannot silently retrain or leak future data.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
root=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
results=json.loads((root/'artifacts/tables/results.json').read_text())

In [2]:
data=pd.read_csv(root/'artifacts/tables/simulation.csv')
display(data.head(12))
display(data.describe(include='all').T)

,strategy,capacity,contacts,contact_cost,assumed_success,incentive,assumed_margin,predicted_value_at_risk,scenario_retained_revenue,scenario_net_contribution
0,random,0.05,18,1,0.02,0,0.3,2697.915149,53.958303,-1.812509
1,random,0.05,18,1,0.02,10,0.3,2697.915149,53.958303,-181.812509
2,random,0.05,18,1,0.02,25,0.3,2697.915149,53.958303,-451.812509
3,random,0.05,18,1,0.05,0,0.3,2697.915149,134.895757,22.468727
4,random,0.05,18,1,0.05,10,0.3,2697.915149,134.895757,-157.531273
5,random,0.05,18,1,0.05,25,0.3,2697.915149,134.895757,-427.531273
6,random,0.05,18,1,0.10,0,0.3,2697.915149,269.791515,62.937454
7,random,0.05,18,1,0.10,10,0.3,2697.915149,269.791515,-117.062546
8,random,0.05,18,1,0.10,25,0.3,2697.915149,269.791515,-387.062546
9,random,0.05,18,5,0.02,0,0.3,2697.915149,53.958303,-73.812509


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
strategy,324,4,random,81,NaN,NaN,NaN,NaN,NaN,NaN,NaN
capacity,324.0,NaN,NaN,NaN,0.116667,0.062457,0.05,0.05,0.1,0.2,0.2
contacts,324.0,NaN,NaN,NaN,42.0,22.48467,18.0,18.0,36.0,72.0,72.0
contact_cost,324.0,NaN,NaN,NaN,5.333333,3.687482,1.0,1.0,5.0,10.0,10.0
assumed_success,324.0,NaN,NaN,NaN,0.056667,0.033049,0.02,0.02,0.05,0.1,0.1
incentive,324.0,NaN,NaN,NaN,11.666667,10.289915,0.0,0.0,10.0,25.0,25.0
assumed_margin,324.0,NaN,NaN,NaN,0.3,0.0,0.3,0.3,0.3,0.3,0.3
predicted_value_at_risk,324.0,NaN,NaN,NaN,8101.234933,4615.751375,2697.915149,4914.739718,6365.27866,11120.512478,18600.356735
scenario_retained_revenue,324.0,NaN,NaN,NaN,459.06998,404.100516,53.958303,157.925125,301.896566,601.663598,1860.035674
scenario_net_contribution,324.0,NaN,NaN,NaN,-576.279006,616.238896,-2454.639463,-873.504987,-421.878701,-146.479506,486.010702


Interpretation: consult the corresponding report for mathematical definitions, executed estimates and their limits. Never turn an assumed campaign response into an observed result.

# Retention policy comparison
Risk-only, value-only, risk × value and random targeting are compared at 5%, 10%, 20% contact capacity. Random results average 200 seeded without-replacement samples. Priority is p(inactivity) × estimated future gross revenue, a value-at-risk proxy, not expected treatment benefit. Under homogeneous treatment success and equal costs, selecting largest scores maximizes the scenario objective for a fixed contact count (exchange argument). With heterogeneous costs/uplift, estimate incremental benefit and solve a knapsack/constrained optimization instead. Capacity is a ceiling: a negative net scenario should recommend no contact rather than force spending.

Scenario retained revenue = sum(priority) × assumed_success. Scenario net contribution = retained revenue × assumed_margin − contacts × (contact_cost + incentive). Incentives are charged to every contact. Margin is 30%; success takes 2%, 5%, 10%; costs GBP 1/5/10; incentives GBP 0/10/25. None are observed. An illustrative 10%-capacity, GBP 5 contact, GBP 10 incentive and 5% effectiveness slice:
```
  strategy  capacity  contacts  contact_cost  assumed_success  incentive  assumed_margin  predicted_value_at_risk  scenario_retained_revenue  scenario_net_contribution
    random       0.1        36             5             0.05         10             0.3              5450.551841                 272.527592                -458.241722
      risk       0.1        36             5             0.05         10             0.3              6650.035339                 332.501767                -440.249470
     value       0.1        36             5             0.05         10             0.3              5995.340646                 299.767032                -450.069890
risk_value       0.1        36             5             0.05         10             0.3             10804.388539                 540.219427                -377.934172
```
Risk × value can win its own arithmetic objective by construction; this is not evidence of actual treatment superiority. Test heterogeneous response with a randomized pilot, intention-to-treat analysis, holdout controls, confidence intervals, incremental margin and customer experience guardrails. Churn risk is not persuadability. Conditional lost value may differ from unconditional BG/NBD forecasts; evaluate risk/value overlap before production.
